将原始数据清洗、填充空值、去除离群值等

In [1]:
import time
import pandas as pd
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer,IterativeImputer
import os

In [2]:
def clean_data(df, method='mean'):
    """
    清洗DataFrame中的空值，将标签转化为数值
        pd.DataFrame: 缺失值处理后的DataFrame
    """
    df_clean = df.copy()
    df_clean['have_stone'] = df_clean['have_stone'].astype('int')  # 确保标签是整数类型
    # 分离标签列
    target_column = 'have_stone'
    if target_column in df_clean.columns:
        y = df_clean.pop(target_column)
    else:
        y = None

    # 方法一：均值填充
    if method == 'mean':
        imputer_num = SimpleImputer(strategy='mean')
        df_clean = pd.DataFrame(imputer_num.fit_transform(df_clean), columns=df_clean.columns)

    # 方法二：删除有缺失值的行
    elif method == 'drop':
        df_clean.dropna(inplace=True)

    # 方法三：回归填充缺失值
    elif method == 'bayesian':
        imputer = IterativeImputer(max_iter=10, random_state=0)
        df_clean = pd.DataFrame(imputer.fit_transform(df_clean), columns=df_clean.columns)
    else:
        raise ValueError(f"Unsupported method: {method}")
    # 将标签列重新加回去
    if y is not None:
        df_clean[target_column] = y
    return df_clean

In [3]:
#待处理的原始文件
print(os.listdir('data_original/'))

['Train-test-dataset_Ver-electronic-haveoutliers.csv', 'Ver-electronic-haveoutliers-markelectrolyte.csv', 'ver-noelectrolyte-logtransform.csv']


In [10]:
#目前是单次处理， 修改需要处理的filepath，从原始文件复制
def is_already_cleaned(output_file_path):
    """
    检查输出文件是否已经存在。
    如果文件已存在，返回 True；否则返回 False。
    """
    return os.path.exists(output_file_path)

file_path = 'data_original/ver-noelectrolyte-logtransform.csv'
file_name= os.path.basename(file_path)
# 去掉文件的扩展名
file_name_without_extension = os.path.splitext(file_name)[0]
print(f"Processing file: {file_name}")
df = pd.read_csv(file_path)

# 确保输出目录存在
output_dir = 'data_cleaned'
os.makedirs(output_dir, exist_ok=True)

# 定义填充方法和对应的文件名
drop_name= f'{file_name_without_extension}_drop.csv'
mean_name= f'{file_name_without_extension}_mean.csv'
bayesian_name= f'{file_name_without_extension}_bayesian.csv'
methods = {
    'drop': drop_name,
    'mean': mean_name,
    'bayesian': bayesian_name,
}

# 使用循环进行数据清洗和保存
for method, output_file_name in methods.items():
    # 拼接完整的输出文件路径
    output_file_path = os.path.join(output_dir, output_file_name)
    # 检查是否已清洗
    if is_already_cleaned(output_file_path):
        print(f"File '{output_file_path}' already cleaned. Skipping...")
        continue
    start_time = time.time()
    df_cleaned = clean_data(df, method=method)
    df_cleaned.to_csv(output_file_path, index=False)
    print(f"Data cleaned using {method} method and saved to '{output_file_path}'. Time taken: {time.time() - start_time:.2f} seconds")

Processing file: ver-noelectrolyte-logtransform.csv
File 'data_cleaned\ver-noelectrolyte-logtransform_drop.csv' already cleaned. Skipping...
File 'data_cleaned\ver-noelectrolyte-logtransform_mean.csv' already cleaned. Skipping...


d:\Cache\Conda\envs\EHR\lib\site-packages\sklearn\impute\_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


Data cleaned using bayesian method and saved to 'data_cleaned\ver-noelectrolyte-logtransform_bayesian.csv'. Time taken: 2063.37 seconds
